# Install Library

In [ ]:
!pip install pandas numpy scikit-learn nltk gensim gradio

# Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from gensim.models import Word2Vec

import gradio as gr

# Download NLTK Resource

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv('//content/drive/MyDrive/Praktikum rill/NLP/UAS/final_merge_dataset.csv')
df.head()

,Judul,Waktu,Link,Content,tag1,tag2,tag3,tag4,tag5,source
0,"Viral Isu PHK Buruh Gudang Garam, Said Iqbal: ...",6 September 2025,https://nasional.kompas.com/read/2025/09/06/14...,"JAKARTA, KOMPAS.com – Presiden Konfederasi Se...",Said Iqbal,industri rokok,PT Gudang Garam,PHK massal,phk massal 2025 terbaru,kompas
1,"Gempa M 5,3 Guncang Pulau Doi Maluku Utara","Senin, 12 Agu 2024 21:58 WIB",https://news.detik.com/berita/d-7486691/gempa-...,"Gempa bumi berkekuatan magnitudo (M) 5,3 mengg...",pulau doi,gempa,NaN,NaN,NaN,detik
2,"Toko Emas Palsu di Riau Dibongkar Polisi, Perh...","Rabu, 30 Jul 2025 22:22 WIB",https://news.detik.com/melindungi-tuah-marwah/...,Satreskrim Polres Bengkalis membongkar praktik...,pemalsuan emas,emas palsu,polres bengkalis,polda riau,melindungi tuah marwah,detik
3,Minyakita Tak Sesuai Ukuran juga Ditemukan di ...,"Senin, 10 Mar 2025 23:15 WIB",https://news.detik.com/berita/d-7816829/minyak...,Polisi mendatangi salah satu gudang Minyakita ...,minyakita,kudus,NaN,NaN,NaN,detik
4,"Pimpin LDP, Sanae Takaichi Calon Kuat PM Perem...",4 Oktober 2025 | 14.00 WIB,https://www.tempo.co/internasional/pimpin-ldp-...,"Baca berita dengan sedikit iklan, klik di sin...",jepang,perdana-menteri,sanae-takaichi,perempuan,ldp,tempo


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Praktikum rill/NLP/UAS/final_merge_dataset.csv')

# rapihin nama kolom
df.columns = df.columns.str.strip().str.lower()

# ambil kolom
df = df[['judul','content','tag1']]

# bikin text & label
df['text'] = df['judul'] + " " + df['content']
df['label'] = df['tag1']

df = df[['text','label']]
df.dropna(inplace=True)

df = df.sample(n=10000, random_state=42)

# Text Preprocessing

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('indonesian'))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

# Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

## FEATURE EXTRACTION 1: TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## FEATURE EXTRACTION 2: Word2Vec

In [ ]:
# tokenisasi ulang
tokenized_text = [text.split() for text in df['clean_text']]

w2v_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=2, workers=4)

def document_vector(doc):
    words = doc.split()
    vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X_train_w2v = np.array([document_vector(text) for text in X_train])
X_test_w2v = np.array([document_vector(text) for text in X_test])

# MODELING (3 MODEL)

## Function Evaluasi

In [ ]:
def evaluate(y_test, y_pred, name="Model"):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1-score:", f1_score(y_test, y_pred, average='weighted'))

## TF-IDF + Model

In [ ]:
# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
evaluate(y_test, y_pred_nb, "TF-IDF + Naive Bayes")

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
evaluate(y_test, y_pred_lr, "TF-IDF + Logistic Regression")

# SVM
svm = SVC()
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
evaluate(y_test, y_pred_svm, "TF-IDF + SVM")


TF-IDF + Naive Bayes
Accuracy: 0.0645
Precision: 0.027677560567755905
Recall: 0.0645
F1-score: 0.023434761469086348


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



TF-IDF + Logistic Regression
Accuracy: 0.1205
Precision: 0.057005952968039085
Recall: 0.1205
F1-score: 0.06301696516038996


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



TF-IDF + SVM
Accuracy: 0.131
Precision: 0.09054753460962481
Recall: 0.131
F1-score: 0.09032487041215835


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Word2Vec + Model

In [ ]:
# Naive Bayes (pakai GaussianNB karena data numerik dense)
from sklearn.naive_bayes import GaussianNB

nb_w2v = GaussianNB()
nb_w2v.fit(X_train_w2v, y_train)
y_pred_nb_w2v = nb_w2v.predict(X_test_w2v)
evaluate(y_test, y_pred_nb_w2v, "Word2Vec + Naive Bayes")

# Logistic Regression
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)
y_pred_lr_w2v = lr_w2v.predict(X_test_w2v)
evaluate(y_test, y_pred_lr_w2v, "Word2Vec + Logistic Regression")

# SVM
svm_w2v = SVC()
svm_w2v.fit(X_train_w2v, y_train)
y_pred_svm_w2v = svm_w2v.predict(X_test_w2v)
evaluate(y_test, y_pred_svm_w2v, "Word2Vec + SVM")


Word2Vec + Naive Bayes
Accuracy: 0.125
Precision: 0.0842219161394316
Recall: 0.125
F1-score: 0.0906351958847379


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Word2Vec + Logistic Regression
Accuracy: 0.143
Precision: 0.07087640048367945
Recall: 0.143
F1-score: 0.08505622460788925


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Word2Vec + SVM
Accuracy: 0.0945
Precision: 0.029456566410090194
Recall: 0.0945
F1-score: 0.03858466743275763


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# DEPLOYMENT (Gradio)

In [ ]:
def predict_text(text):
    clean = preprocess(text)
    vector = tfidf.transform([clean])
    pred = lr.predict(vector)
    return pred[0]

interface = gr.Interface(
    fn=predict_text,
    inputs="text",
    outputs="text",
    title="Sentiment Analysis"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c011771f129a1912d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
